# Module 1.4 — Generator Metrics: Reference-Based

Module 1.3's three metrics all judge the answer on its own terms — grounded, relevant, consistent with a trusted context. None of them ask the most direct question: **is the answer actually correct**, against a known-good reference? That's what this notebook covers, plus the tool you reach for whenever a stock metric doesn't fit your exact rubric: a custom `GEval`.

_Source: adapted from `RAG_Evaluation/3.Custom_LLM_as_a_Judge _(G-Eval).ipynb` (the `evaluation_steps`-based `GEval` pattern) and `04_Agent_RAG_Eval/rag_agent_eval_langgraph_openai.ipynb` Part 2b (Answer Correctness via `criteria`-based `GEval`, and the embedding-based Answer Semantic Similarity helper) — both adapted to standalone, hand-constructed test cases.

## Why these need a reference — and what that costs

Recall Module 0 §2's reference-based/referenceless split: **Answer Correctness has no dedicated DeepEval metric class**, unlike Faithfulness or Relevancy. That's not an oversight — "is it factually right" is inherently specific to what "right" means for your task, so DeepEval's own recommended pattern is to write a custom `GEval` rubric rather than ship one generic class. **Answer Semantic Similarity** isn't a DeepEval class at all — it's plain embedding cosine similarity, cheap and fast with no LLM judge call, but weaker at catching subtle factual errors (two answers can be semantically close while disagreeing on a key number or fact).

Both require an `expected_output` — a reference answer someone had to write. That's the cost side of the reference-based/referenceless tradeoff from Module 0: precise, but only as scalable as your ability to keep reference answers curated and current.

### Two ways to write a `GEval` rubric: `criteria` vs. `evaluation_steps`

In [ ]:
# ============ ANSWER CORRECTNESS (GEval, criteria-based) ============
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval

query = "What is the time complexity of deleting a node from a BST?"
reference_answer = "Deleting a node from a BST takes O(h) time, where h is the height of the tree."

correct_case = LLMTestCase(
    input=query,
    actual_output="Deleting a node from a binary search tree takes O(h) time, h being the tree's height.",
    expected_output=reference_answer,
)

wrong_case = LLMTestCase(
    input=query,
    actual_output="Deleting a node from a BST takes O(1) time since deletion is a constant-time pointer update.",
    expected_output=reference_answer,
)

# `criteria` -- a short, plain-language description; DeepEval expands it into evaluation steps internally.
answer_correctness = GEval(
    name="Answer Correctness",
    criteria="Determine whether the actual output is factually correct relative to the expected output. "
             "Focus on whether the key facts (numbers, complexities, entities) match -- wording may differ.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    model="gpt-4o",
    threshold=0.5,
)

for label, case in [("Correct answer (different wording, same fact)", correct_case), ("Wrong answer (wrong complexity)", wrong_case)]:
    answer_correctness.measure(case)
    print(f"{label}: score={answer_correctness.score:.2f}  reason={answer_correctness.reason}\n")

In [ ]:
# ============ CUSTOM GEval (evaluation_steps-based) ============
# `evaluation_steps` -- explicit, ordered instructions instead of one criteria sentence.
# NOTE: you can only provide either `criteria` or `evaluation_steps`, never both.
rag_context = [
    "A binary search tree (BST) is a tree data structure where each node has at most two children.",
    "Deleting a node from a BST takes O(h) time, where h is the height of the tree.",
]

fact_checker = GEval(
    name="RAG Fact Checker",
    evaluation_steps=[
        "Create a list of statements from 'actual output'",
        "Validate if they are relevant and answer the given question in 'input', penalize if any statements are irrelevant",
        "Also validate if they are consistent with 'expected output', penalize if any statements are missing or factually wrong",
        "Also validate if these statements are grounded in the 'retrieval context', penalize if they are missing or factually wrong",
        "Finally also penalize if any statements seem to be invented or made up and do not make sense given 'input' and 'retrieval context'",
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT, LLMTestCaseParams.RETRIEVAL_CONTEXT,
    ],
    model="gpt-4o",
    threshold=0.5,
)

fabricated_case = LLMTestCase(
    input=query,
    actual_output="Deleting a node from a BST takes O(h) time, and BSTs automatically rebalance themselves on every deletion.",
    expected_output=reference_answer,
    retrieval_context=rag_context,
)

fact_checker.measure(fabricated_case)
print(f"RAG Fact Checker: score={fact_checker.score:.2f}  reason={fact_checker.reason}")

**Reading the output:** `evaluation_steps` is the more controllable form — instead of trusting DeepEval to expand a one-line `criteria` into a good evaluation procedure, you write the procedure yourself, step by step, and can weave in multiple test-case fields (here: relevance to `input`, agreement with `expected_output`, *and* grounding in `retrieval_context`, all in one rubric). This is the same tool as the `criteria`-based Answer Correctness above — `GEval` — just used with the more explicit of its two configuration styles. Reach for `evaluation_steps` whenever a single-sentence criteria doesn't give you enough control over exactly what gets penalized.

### Answer Semantic Similarity — embedding cosine similarity, no LLM judge

In [ ]:
# ============ ANSWER SEMANTIC SIMILARITY ============
import numpy as np
from langchain_openai import OpenAIEmbeddings

embedder = OpenAIEmbeddings(model="text-embedding-3-small")


def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


for label, actual in [
    ("Correct answer (different wording, same fact)", correct_case.actual_output),
    ("Wrong answer (wrong complexity)", wrong_case.actual_output),
]:
    vec_actual, vec_reference = embedder.embed_documents([actual, reference_answer])
    score = cosine_similarity(vec_actual, vec_reference)
    print(f"{label}: cosine similarity={score:.3f}")

**Reading the output:** watch for the exact gap Module 0 warned about — the wrong-complexity answer ("O(1)... constant-time") is topically about the same subject, using similar vocabulary, so its embedding similarity to the reference may come back higher than you'd expect for an answer that's factually wrong. This is Answer Semantic Similarity's known weakness: it measures *topical* closeness, not *factual* agreement, which is exactly why it's paired with (not a replacement for) an LLM-judged Answer Correctness check when you actually need to catch a wrong number or fact.

## Summary

- **Answer Correctness** has no dedicated DeepEval class — write a custom `GEval` with `criteria` (quick) or `evaluation_steps` (more controllable, and able to combine multiple test-case fields into one rubric).
- **Answer Semantic Similarity** is cheap embedding cosine similarity with no LLM call — fast, but only measures topical closeness, not factual correctness.
- Both need `expected_output` — the reference-based cost from Module 0 §2 applies directly: precise, but only as scalable as your ability to write and maintain reference answers.
- This closes out the individual-metric modules. [Module 1.5](05_RAG_Eval_Inside_the_Pipeline.ipynb) shows these same ideas wired directly into a RAG pipeline as evaluation *gates*, not just after-the-fact scoring; [Module 1.7](07_RAG_Capstone_Build_and_Evaluate.ipynb) runs the full suite — retrieval and generation, reference-based and referenceless — against a real RAG system and a synthetic golden dataset.